# TP Final Integrador — Predicción de Cancelaciones Amazon


In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:
# ============================================================
# 1. CARGA DE DATOS
# ============================================================
df = pd.read_csv('Amazon Sale Report.csv')

# Limpiar nombres de columnas (eliminar espacios trailing/leading)
df.columns = df.columns.str.strip()

print(f"Shape original: {df.shape}")
print("\nPrimeras filas:")
display(df.head())

In [ ]:
# ============================================================
# 2. EXPLORACIÓN INICIAL
# ============================================================
print("Columnas:")
print(df.columns.tolist())
print("\nValores nulos:")
print(df.isnull().sum())
print("\nDistribución de Status:")
print(df['Status'].value_counts())

In [ ]:
# ============================================================
# 3. CREAR VARIABLE OBJETIVO
# ============================================================
df['cancelado'] = df['Status'].apply(lambda x: 1 if 'Cancelled' in str(x) else 0)

print("Distribución de la variable objetivo:")
print(df['cancelado'].value_counts())
print(f"\nPorcentaje de cancelaciones: {df['cancelado'].mean():.2%}")

In [ ]:
# ============================================================
# 4. LIMPIEZA Y FEATURE ENGINEERING
# ============================================================

# 4a. Eliminar columnas innecesarias y con data leakage ---
cols_to_drop = [
    'index', 'Order ID', 'SKU', 'ASIN',
    'promotion-ids', 'fulfilled-by', 'Unnamed: 22',
    'Status',                # ya usada para crear target
    'Amount',                # data leakage (0 en cancelados)
    'Courier Status',        # data leakage (Unknown en cancelados)
    'currency',              # columna constante (solo INR)
]

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 4b. Tratamiento de nulos ---
cols_fill = ['ship-city', 'ship-state', 'ship-postal-code', 'ship-country']
for col in cols_fill:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# 4c. Fechas → Features ---
df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y', errors='coerce')
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day
df['day_of_week'] = df['Date'].dt.dayofweek
df = df.drop(columns=['Date'])

# 4d. Check final ---
print(f"Shape final: {df.shape}")
print("\nNulos restantes:")
print(df.isnull().sum())

In [ ]:
# ============================================================
# 5. PREPARACIÓN DE FEATURES Y TARGET
# ============================================================
X = df.drop('cancelado', axis=1)
y = df['cancelado']

# 5a. Eliminar columnas de alta cardinalidad / bajo valor predictivo ---
cols_drop_extra = ['ship-city', 'ship-postal-code', 'Style']
X = X.drop(columns=[col for col in cols_drop_extra if col in X.columns])

# 5b. Identificar columnas numéricas y categóricas ---
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Columnas numéricas ({len(num_cols)}): {num_cols}")
print(f"Columnas categóricas ({len(cat_cols)}): {cat_cols}")

In [ ]:
# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"\nDistribución en train:\n{y_train.value_counts(normalize=True)}")
print(f"\nDistribución en test:\n{y_test.value_counts(normalize=True)}")

In [ ]:
# ============================================================
# 7. CREAR PIPELINES CON PREPROCESAMIENTO
# ============================================================

# Preprocesador: escalar numéricas + one-hot categóricas ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Pipeline Logistic Regression con class_weight balanceado ---
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Pipeline Random Forest con class_weight balanceado ---
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

print("Pipelines creados exitosamente.")
print("\nPipeline LR:")
print(lr_pipeline)
print("\nPipeline RF:")
print(rf_pipeline)

In [ ]:
# ============================================================
# 8. ENTRENAMIENTO Y EVALUACIÓN
# ============================================================

# Entrenar modelos ---
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# Predicciones ---
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# Métricas ---
print("=== Logistic Regression (class_weight=balanced) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_prob_lr):.4f}")
print(classification_report(y_test, y_pred_lr))

print("\n=== Random Forest (class_weight=balanced) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_prob_rf):.4f}")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# ============================================================
# 9. VALIDACIÓN CRUZADA (5-FOLD ESTRATIFICADO)
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv_auc = cross_val_score(lr_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
rf_cv_auc = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print("=== Validación Cruzada 5-Fold (AUC) ===")
print(f"Logistic Regression: {lr_cv_auc.mean():.4f} (+/- {lr_cv_auc.std():.4f})")
print(f"Random Forest:       {rf_cv_auc.mean():.4f} (+/- {rf_cv_auc.std():.4f})")

In [ ]:
# ============================================================
# 10. CURVA ROC — AMBOS MODELOS
# ============================================================
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Comparación de Modelos")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 11. MATRICES DE CONFUSIÓN — AMBOS MODELOS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['No Cancelado', 'Cancelado'])
disp_lr.plot(ax=axes[0], cmap='Blues')
axes[0].set_title("Logistic Regression")

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['No Cancelado', 'Cancelado'])
disp_rf.plot(ax=axes[1], cmap='Greens')
axes[1].set_title("Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 12. IMPORTANCIA DE VARIABLES — AMBOS MODELOS
# ============================================================

# Obtener nombres de features después del OneHotEncoder ---
ohe = lr_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names

# Logistic Regression: coeficientes estandarizados ---
lr_coefs = lr_pipeline.named_steps['classifier'].coef_[0]
lr_importance = pd.DataFrame({
    'feature': all_feature_names,
    'coef': lr_coefs,
    'abs_coef': np.abs(lr_coefs)
}).sort_values('abs_coef', ascending=False)

# Random Forest: feature importance nativa ---
rf_importances = rf_pipeline.named_steps['classifier'].feature_importances_
rf_importance = pd.DataFrame({
    'feature': all_feature_names,
    'importance': rf_importances
}).sort_values('importance', ascending=False)

# Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_lr = lr_importance.head(10)
axes[0].barh(top_lr['feature'], top_lr['coef'], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_title("Top 10 Coeficientes — Logistic Regression")
axes[0].set_xlabel("Coeficiente (datos escalados)")

top_rf = rf_importance.head(10)
axes[1].barh(top_rf['feature'], top_rf['importance'], color='darkorange')
axes[1].invert_yaxis()
axes[1].set_title("Top 10 Feature Importance — Random Forest")
axes[1].set_xlabel("Importancia")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 13. RESUMEN FINAL
# ============================================================
print("=" * 60)
print("RESUMEN FINAL DE MODELOS")
print("=" * 60)
print(f"\n{'Métrica':<20} {'Logistic Reg.':<18} {'Random Forest':<18}")
print("-" * 60)
print(f"{'Accuracy (test)':<20} {accuracy_score(y_test, y_pred_lr):<18.4f} {accuracy_score(y_test, y_pred_rf):<18.4f}")
print(f"{'AUC (test)':<20} {roc_auc_score(y_test, y_prob_lr):<18.4f} {roc_auc_score(y_test, y_prob_rf):<18.4f}")
print(f"{'AUC (CV 5-fold)':<20} {lr_cv_auc.mean():<18.4f} {rf_cv_auc.mean():<18.4f}")
print("=" * 60)
print(f"\nMejor modelo por AUC en test: {'Logistic Regression' if roc_auc_score(y_test, y_prob_lr) > roc_auc_score(y_test, y_prob_rf) else 'Random Forest'}")
print(f"Mejor modelo por CV 5-fold:     {'Logistic Regression' if lr_cv_auc.mean() > rf_cv_auc.mean() else 'Random Forest'}")